<a href="https://colab.research.google.com/github/celinehsu/Complete-Python-3-Bootcamp/blob/master/FindSimilarJobs_(Careers_Gov).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install required packages
!pip install -q pandas requests sentence-transformers scikit-learn

import requests
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

# ===========================================================================
# 2. PASTE YOUR TARGET JOB DESCRIPTION HERE:
# ===========================================================================
MY_JOB_DESCRIPTION = """
The Senior Manager in Centre for Information Management (CIM) will lead a team to support DSTA’s record management policies, processes and systems in accordance to Government Instruction Manual on Records Management and DSTA records management framework, as well as drive process reviews and digitalization to ensure that DSTA’s record management system continue meet DSTA’s evolving needs.

Job scope:
•	Lead and manage DSTA Enterprise Content Management and records repositories to comply with DSTA Governance Framework and align with international and national standards and practices for optimal records management.
•	Plan and develop DSTA Records Management Roadmap for digital/digitised records including storage, accessibility and disposal.
•	Formalise holistic governance framework, policies and processes for continual improvement in management of DSTA enterprise content (i.e. digital/digitised records). Evaluate, test and implement the system capabilities.
•	Review, recommend and drive digitalisation initiatives with key stakeholders (e.g. DSTA management/entities, MINDEF/SAF, external agencies) in records management to meet organisation's evolving business needs.
•	Support DSTA Chief Record Officer in Records Management to ensure compliance with Government’s Instruction Manual requirements.


"""
# ===========================================================================

# 3. How many matching jobs do you want to download?
HOW_MANY_MATCHES = 50  # You can change this to 10, 30, 50, etc.

print("1. Downloading latest active listings from Careers@Gov...")
url = "https://raw.githubusercontent.com/opengovsg/careersgovsg-jobs-data/main/data/job-listings.json"
response = requests.get(url)
jobs = response.json()
df = pd.DataFrame(jobs)

# Helper function to generate direct job URLs
def get_url(row):
    plat, jid, pno = row.get("platform", ""), row.get("jobId", ""), row.get("postingNo", "")
    if plat == "hrp":
        return f"https://jobs.careers.gov.sg/jobs/{plat}/{jid}/{pno}"
    elif plat == "greenhouse":
        return f"https://jobs.careers.gov.sg/jobs/{plat}/{jid}?gh_jid={jid}"
    elif plat == "workable":
        return f"https://jobs.careers.gov.sg/jobs/{plat}/{jid}/{pno}"
    return "https://jobs.careers.gov.sg"

# Clean HTML tags from job descriptions
def clean_html(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<br\s*/?>", "\n", text)
    text = re.sub(r"</li>", "\n", text)
    clean = re.sub(r"<.*?>", " ", text)
    return "\n".join([line.strip() for line in clean.splitlines() if line.strip()])

df["Job_Link"] = df.apply(get_url, axis=1)
df["Clean_Responsibilities"] = df.get("jobResponsibilities", pd.Series([""]*len(df))).fillna("").apply(clean_html)
df["Clean_Requirements"] = df.get("jobRequirements", pd.Series([""]*len(df))).fillna("").apply(clean_html)
df["Clean_Description"] = df.get("jobDescription", pd.Series([""]*len(df))).fillna("").apply(clean_html)

# Create full consolidated description for each job
def build_full_jd(row):
    sections = []
    if row["Clean_Description"]:
        sections.append(f"--- OVERVIEW ---\n{row['Clean_Description']}")
    if row["Clean_Responsibilities"]:
        sections.append(f"--- RESPONSIBILITIES ---\n{row['Clean_Responsibilities']}")
    if row["Clean_Requirements"]:
        sections.append(f"--- REQUIREMENTS ---\n{row['Clean_Requirements']}")
    return "\n\n".join(sections)

df["Full_Job_Description"] = df.apply(build_full_jd, axis=1)

# Combined text for similarity matching
df["matching_text"] = (
    "Title: " + df["jobTitle"].fillna("") + "\n" +
    "Agency: " + df["agency"].fillna("") + "\n" +
    df["Full_Job_Description"]
)

print(f"2. Loaded {len(df)} postings. Loading AI model and ranking matches...")
model = SentenceTransformer("all-MiniLM-L6-v2")

target_emb = model.encode([MY_JOB_DESCRIPTION])
job_embs = model.encode(df["matching_text"].tolist(), show_progress_bar=True)

df["Match_Score_%"] = (cosine_similarity(target_emb, job_embs)[0] * 100).round(1)

# Sort and select top matching jobs
top_matches = (
    df.sort_values(by="Match_Score_%", ascending=False)
    .head(HOW_MANY_MATCHES)
    .reset_index(drop=True)
)

# Prepare clean export columns
export_df = pd.DataFrame({
    "Rank": range(1, len(top_matches) + 1),
    "Match Score (%)": top_matches["Match_Score_%"],
    "Job Title": top_matches["jobTitle"],
    "Agency": top_matches["agency"],
    "Job Link": top_matches["Job_Link"],
    "Full Job Description": top_matches["Full_Job_Description"]
})

# Save to CSV with UTF-8 BOM so Excel opens it cleanly without scrambled characters
filename = "matched_careersgov_jobs.csv"
export_df.to_csv(filename, index=False, encoding="utf-8-sig")

print("\n" + "="*80)
print(f"DONE! Top {len(export_df)} matches found.")
print("="*80)

for _, row in export_df.head(5).iterrows():
    print(f"{row['Rank']}. [{row['Match Score (%)']}% Match] {row['Job Title']} ({row['Agency']})")
    print(f"   Link: {row['Job Link']}\n")

print("Downloading file to your computer now...")
files.download(filename)

1. Downloading latest active listings from Careers@Gov...
2. Loaded 2105 postings. Loading AI model and ranking matches...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/66 [00:00<?, ?it/s]


DONE! Top 50 matches found.
1. [97.9000015258789% Match] Senior Manager, Centre for Info Mgmt (CIM), Corp Planning & Services (Defence Science and Technology Agency)
   Link: https://jobs.careers.gov.sg/jobs/hrp/18124292/005056a3-d347-1fe1-b0c4-843f00cfc2bf

2. [72.5999984741211% Match] (Senior) Business Partner, Administration (Enterprise Singapore)
   Link: https://jobs.careers.gov.sg/jobs/hrp/18198326/005056a3-d347-1fe1-b5f8-853b256b42c5

3. [69.0999984741211% Match] Senior Data Manager (Singapore Land Authority)
   Link: https://jobs.careers.gov.sg/jobs/hrp/17921981/005056a3-d347-1fe1-afd3-dd49543082bf

4. [68.30000305175781% Match] Deputy/Assistant Director, Data Governance officer (Contract) (Monetary Authority of Singapore)
   Link: https://jobs.careers.gov.sg/jobs/hrp/17994910/005056a3-d347-1fe1-b5ad-0fe22c2682c5

5. [67.0999984741211% Match] Senior Assistant Data Manager / Data Manager (Singapore Land Authority)
   Link: https://jobs.careers.gov.sg/jobs/hrp/18161973/005056a3-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>